### Project: Document Flow — Lightweight Document Ingestion & RAG Pipeline

##### Stage 1 — Ingestion Pipeline (Python + SQL)

In [ ]:
import boto3
import hashlib
import os
from pathlib import Path
from dotenv import load_dotenv
import psycopg2
import fitz
import io
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
import anthropic

In [ ]:
load_dotenv("rag_s3.env")
file_path = #path for file
bucket = #bucket name

s3 = boto3.client(
    "s3",
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    region_name=os.getenv("AWS_REGION")
)

##### Create a unique fingerprint for the document
doc_id = hashlib.md5(Path(file_path).read_bytes()).hexdigest()
s3.upload_file(file_path, bucket, f"raw/{doc_id}.pdf")

doc_id = hashlib.md5(Path(file_path).read_bytes()).hexdigest()

conn = psycopg2.connect(
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT"),
    dbname=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD")
)

cursor = conn.cursor()
cursor.execute("""
    INSERT INTO documents (doc_id, source_path, status, ingested_at)
    VALUES (%s, %s, 'raw', NOW())
    ON CONFLICT (doc_id) DO NOTHING
""", (doc_id, file_path))
conn.commit()
conn.close()
print("Document Log Recorded")

#### Stage 2 — Chunking + Embedding

In [ ]:
# Pull from S3 into memory — never touches your hard drive
response = s3.get_object(Bucket=bucket, Key=f"raw/{doc_id}.pdf")
pdf_bytes = response["Body"].read()

# Open directly from bytes — no local file needed
doc = fitz.open(stream=pdf_bytes, filetype="pdf")
text = ""
for page in doc:
    text += page.get_text()

print(text[:500])

def chunk_text(text: str, chunk_size=500, overlap=100):
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(text)
print(f"{len(chunks)} chunks created")
print(chunks[0])
print("---")
print(chunks[1])

# Load embedding model locally
model = SentenceTransformer("all-MiniLM-L6-v2")

# Connect to Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("docflow")

# Embed and store each chunk
vectors = []
for i, chunk in enumerate(chunks):
    embedding = model.encode(chunk).tolist()
    vectors.append({
        "id": f"{doc_id}_chunk_{i}",
        "values": embedding,
        "metadata": {"doc_id": doc_id, "chunk_index": i, "text": chunk}
    })

# Upload all vectors to Pinecone
index.upsert(vectors=vectors)
print(f"{len(vectors)} chunks uploaded to Pinecone")

#### Stage 3 -- RAG Query Layer

In [ ]:
def query_documents(question: str):
    query_embedding = model.encode(question).tolist()
    
    # Search Pinecone for relevant chunks
    results = index.query(
        vector=query_embedding,
        top_k=5,
        include_metadata=True
    )
    
    # Extract text from results
    chunks = [match["metadata"]["text"] for match in results["matches"]]
    context = "\n\n".join(chunks)
    
    # Send to Claude with context
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": f"""Use only the context below to answer the question.
            
Context:
{context}

Question: {question}"""
        }]
    )
    
    return response.content[0].text

# Test it
answer = query_documents("What does the report say about Gen Z mental health?")
print(answer)